# Triton: raw pointer ↔ `tl.make_block_ptr` 1:1 교환 가능성 사례 모음

**핵심 결론:** 항상 1:1 교환은 안 된다.

- ✅ **표현 가능 + 등가**: 단일 텐서의 contiguous tile load/store, boundary 처리
- ⚠️ **표현 가능하지만 별도 처리 필요**: mid-tensor algorithmic mask (boundary_check로 대체 불가, `tl.where` 분리 필수)
- ❌ **직접 1:1 변환 불가**: 한 tile 안에서 행마다 base가 다른 gather (`indices[:, None] * stride + ...`). 우회 가능하지만 직렬화로 성능 저하
- 🤷 **변환 의미 없음**: 스칼라 1개 로드 — block_ptr 추상화 비용만 늘어남

각 사례에서 raw 버전과 block_ptr 버전을 나란히 두고 (a) 컴파일 가능성, (b) 결과 동등성, (c) 표현 비용을 비교한다.

## 환경
- GPU 필수 (Triton은 CUDA 백엔드 가정)
- 이 노트북은 GPU가 없으면 각 셀이 안전하게 skip된다.

In [ ]:
import torch
import triton
import triton.language as tl

HAS_CUDA = torch.cuda.is_available()
print(f"CUDA available: {HAS_CUDA}")
if HAS_CUDA:
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Triton version: {triton.__version__}")

def skip_if_no_cuda():
    if not HAS_CUDA:
        print("⏭  GPU 없음 — 셀 skip")
        return True
    return False

## 사례 1 — ✅ 1:1 교환 가능 (1-D contiguous tile + boundary)

가장 단순한 케이스. 1-D 벡터를 BLOCK 단위로 나눠 합 계산. 마지막 program이 partial tile을 다룬다 (`N % BLOCK != 0`).

raw: `tl.load(X + offs, mask=offs < N, other=0.0)`
block: `tl.load(X_ptr, boundary_check=(0,), padding_option="zero")`

둘 다 boundary 밖을 0으로 채우므로 **결과가 정확히 같다.**

In [ ]:
@triton.jit
def block_sum_raw(X, Out, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    x = tl.load(X + offs, mask=mask, other=0.0)
    tl.store(Out + pid, tl.sum(x))

@triton.jit
def block_sum_blockptr(X, Out, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    X_ptr = tl.make_block_ptr(
        base=X,
        shape=(N,),
        strides=(1,),
        offsets=(pid * BLOCK,),
        block_shape=(BLOCK,),
        order=(0,),
    )
    x = tl.load(X_ptr, boundary_check=(0,), padding_option="zero")
    tl.store(Out + pid, tl.sum(x))

if not skip_if_no_cuda():
    N, BLOCK = 1000, 128                              # N % BLOCK != 0 — partial tile
    x = torch.randn(N, device="cuda", dtype=torch.float32)
    grid = (triton.cdiv(N, BLOCK),)
    out_raw = torch.empty(grid[0], device="cuda")
    out_blk = torch.empty(grid[0], device="cuda")
    block_sum_raw[grid](x, out_raw, N, BLOCK=BLOCK)
    block_sum_blockptr[grid](x, out_blk, N, BLOCK=BLOCK)
    print(f"raw  result : {out_raw.tolist()[:3]} ... last={out_raw[-1].item():.4f}")
    print(f"blk  result : {out_blk.tolist()[:3]} ... last={out_blk[-1].item():.4f}")
    print(f"max abs err : {(out_raw - out_blk).abs().max().item()}  ← 0이면 1:1 동등")

## 사례 2 — ✅ 1:1 교환 가능 (2-D matrix copy + 양축 boundary)

2-D contiguous tile copy. 두 축 모두 boundary가 발생할 수 있다 (`M % BLOCK_M != 0`, `N % BLOCK_N != 0`).

raw: `mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)`
block: `boundary_check=(0, 1)` — 두 축 동시 처리

역시 1:1 등가. block_ptr이 약간 더 짧은 표현.

In [ ]:
@triton.jit
def copy_2d_raw(X, Y, M, N, sx_m, sx_n, sy_m, sy_n,
                BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    offs_m = pm * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pn * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    x = tl.load(X + offs_m[:, None] * sx_m + offs_n[None, :] * sx_n,
                mask=mask, other=0.0)
    tl.store(Y + offs_m[:, None] * sy_m + offs_n[None, :] * sy_n,
             x, mask=mask)

@triton.jit
def copy_2d_blockptr(X, Y, M, N, sx_m, sx_n, sy_m, sy_n,
                     BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    X_ptr = tl.make_block_ptr(
        base=X, shape=(M, N), strides=(sx_m, sx_n),
        offsets=(pm * BLOCK_M, pn * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    Y_ptr = tl.make_block_ptr(
        base=Y, shape=(M, N), strides=(sy_m, sy_n),
        offsets=(pm * BLOCK_M, pn * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    x = tl.load(X_ptr, boundary_check=(0, 1), padding_option="zero")
    tl.store(Y_ptr, x, boundary_check=(0, 1))

if not skip_if_no_cuda():
    M, N, BLOCK_M, BLOCK_N = 100, 70, 32, 32          # 양축 모두 partial 발생
    x = torch.randn(M, N, device="cuda", dtype=torch.float32)
    y_raw = torch.zeros_like(x)
    y_blk = torch.zeros_like(x)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    copy_2d_raw[grid](x, y_raw, M, N, x.stride(0), x.stride(1),
                      y_raw.stride(0), y_raw.stride(1),
                      BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    copy_2d_blockptr[grid](x, y_blk, M, N, x.stride(0), x.stride(1),
                           y_blk.stride(0), y_blk.stride(1),
                           BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    print(f"raw vs ref : {(y_raw - x).abs().max().item()}  (0이면 정확)")
    print(f"blk vs ref : {(y_blk - x).abs().max().item()}  (0이면 정확)")
    print(f"raw vs blk : {(y_raw - y_blk).abs().max().item()}  ← 0이면 1:1 동등")

## 사례 3 — ⚠️ mid-tensor algorithmic mask: 표현은 가능하지만 1:1 아님

**규칙**: 짝수 인덱스의 값만 살리고 홀수는 0으로. 이건 "텐서 끝"의 boundary가 아니라 **알고리즘적 마스크**다.

- raw: `tl.load(X + offs, mask=(offs < N) & (offs % 2 == 0), other=0.0)` — 한 줄에 boundary + algorithmic 둘 다
- block_ptr: `boundary_check`는 텐서 끝만 처리. **짝수 마스크는 별도로 `tl.where`** 필요. 표현 단계가 분리됨.

**핵심**: block_ptr API는 `tl.load(block_ptr, mask=...)` 인자를 **받지 않는다.** algorithmic mask가 필요하면 load 후 `tl.where`로 별도 처리해야 한다. 결과는 같지만 "한 줄 ↔ 두 단계"의 비대칭이 발생.

In [ ]:
@triton.jit
def even_only_raw(X, Out, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    # boundary + algorithmic을 한 mask에 합침
    mask = (offs < N) & (offs % 2 == 0)
    x = tl.load(X + offs, mask=mask, other=0.0)
    tl.store(Out + offs, x, mask=offs < N)

@triton.jit
def even_only_blockptr(X, Out, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    X_ptr = tl.make_block_ptr(
        base=X, shape=(N,), strides=(1,),
        offsets=(pid * BLOCK,), block_shape=(BLOCK,), order=(0,),
    )
    # 1단계: boundary만 처리 (텐서 끝 zero-pad)
    x = tl.load(X_ptr, boundary_check=(0,), padding_option="zero")
    # 2단계: algorithmic mask는 별도 tl.where
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    x = tl.where(offs % 2 == 0, x, 0.0)
    Out_ptr = tl.make_block_ptr(
        base=Out, shape=(N,), strides=(1,),
        offsets=(pid * BLOCK,), block_shape=(BLOCK,), order=(0,),
    )
    tl.store(Out_ptr, x, boundary_check=(0,))

if not skip_if_no_cuda():
    N, BLOCK = 1000, 128
    x = torch.randn(N, device="cuda", dtype=torch.float32)
    out_raw = torch.zeros_like(x)
    out_blk = torch.zeros_like(x)
    grid = (triton.cdiv(N, BLOCK),)
    even_only_raw[grid](x, out_raw, N, BLOCK=BLOCK)
    even_only_blockptr[grid](x, out_blk, N, BLOCK=BLOCK)
    # 참조: torch로 같은 연산
    ref = x.clone()
    ref[1::2] = 0
    print(f"raw vs ref : {(out_raw - ref).abs().max().item()}")
    print(f"blk vs ref : {(out_blk - ref).abs().max().item()}")
    print(f"raw vs blk : {(out_raw - out_blk).abs().max().item()}  ← 결과는 같지만")
    print("→ block_ptr 버전은 load 후 별도 tl.where 단계가 강제됨 (1:1 아님)")

## 사례 4 — ❌ Gather: 한 tile 안 행마다 다른 base — block_ptr 직접 변환 불가

**문제**: `Indices` 라는 [BLOCK_M] int 벡터가 있고, 그것을 **행 인덱스**로 사용해 X의 행을 gather. 즉 한 tile 안에서 **각 행이 X의 다른 위치**에서 온다.

이건 vLLM의 paged KV `block_table` 간접 인덱싱과 똑같은 패턴이다.

- raw: `tl.load(X + idx[:, None] * sx_row + offs_d[None, :] * sx_col, ...)` — **단일 호출**
- block_ptr: `make_block_ptr`은 단일 base + 정적 strides 필수 → `idx[:, None]`을 base에 못 박음. **직접 변환 불가능.**

**우회**: 행마다 fresh `make_block_ptr`을 만드는 직렬 루프. BLOCK_M번 반복 → 한 번의 vectorized gather가 BLOCK_M번의 직렬 load로 풀어짐. **표현은 가능하지만 성능은 명확히 더 나쁘다.**

(vllm_attn/block_ptr의 paged 변형은 이 문제를 다른 방식으로 풀었다 — 한 iteration이 정확히 한 logical block만 다루도록 재구성해서 `idx`를 **스칼라**로 만든 것. gather 자체를 없앤 셈. `BLOCK_PTR_MIGRATION.md` §1 참조.)

In [ ]:
@triton.jit
def gather_rows_raw(X, Indices, Out, NUM_ROWS, D,
                    sx_row, sx_col,
                    BLOCK_M: tl.constexpr, BLOCK_D: tl.constexpr):
    pid = tl.program_id(0)
    offs_m = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)
    m_mask = offs_m < NUM_ROWS
    d_mask = offs_d < D
    idx = tl.load(Indices + offs_m, mask=m_mask, other=0)        # [BLOCK_M] 행 인덱스
    # 한 호출 — 행마다 다른 base (idx[:, None] * sx_row)
    x = tl.load(
        X + idx[:, None] * sx_row + offs_d[None, :] * sx_col,
        mask=m_mask[:, None] & d_mask[None, :], other=0.0,
    )
    tl.store(
        Out + offs_m[:, None] * D + offs_d[None, :],
        x,
        mask=m_mask[:, None] & d_mask[None, :],
    )

@triton.jit
def gather_rows_blockptr_workaround(X, Indices, Out, NUM_ROWS, D,
                                    sx_row, sx_col,
                                    BLOCK_M: tl.constexpr, BLOCK_D: tl.constexpr):
    pid = tl.program_id(0)
    # 행마다 fresh make_block_ptr — BLOCK_M번 직렬 iteration (vectorized gather 대신)
    for i in tl.static_range(BLOCK_M):
        m_idx = pid * BLOCK_M + i
        # 범위 밖 행은 skip 대신 idx=0으로 읽고 store mask로 처리 (단순화)
        in_range = m_idx < NUM_ROWS
        row_idx = tl.load(Indices + m_idx, mask=in_range, other=0)
        X_row_ptr = tl.make_block_ptr(
            base=X + row_idx * sx_row,
            shape=(D,), strides=(sx_col,),
            offsets=(0,), block_shape=(BLOCK_D,), order=(0,),
        )
        x_row = tl.load(X_row_ptr, boundary_check=(0,), padding_option="zero")
        offs_d = tl.arange(0, BLOCK_D)
        tl.store(
            Out + m_idx * D + offs_d,
            x_row,
            mask=in_range & (offs_d < D),
        )

if not skip_if_no_cuda():
    NUM_ROWS, D = 256, 64
    SOURCE_ROWS = 1000
    BLOCK_M, BLOCK_D = 32, 64
    x_src = torch.randn(SOURCE_ROWS, D, device="cuda", dtype=torch.float32)
    indices = torch.randint(0, SOURCE_ROWS, (NUM_ROWS,), device="cuda", dtype=torch.int32)
    out_raw = torch.zeros(NUM_ROWS, D, device="cuda")
    out_blk = torch.zeros(NUM_ROWS, D, device="cuda")
    grid = (triton.cdiv(NUM_ROWS, BLOCK_M),)
    gather_rows_raw[grid](
        x_src, indices, out_raw, NUM_ROWS, D,
        x_src.stride(0), x_src.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_D=BLOCK_D,
    )
    gather_rows_blockptr_workaround[grid](
        x_src, indices, out_blk, NUM_ROWS, D,
        x_src.stride(0), x_src.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_D=BLOCK_D,
    )
    ref = x_src[indices.long()]
    print(f"raw vs ref : {(out_raw - ref).abs().max().item()}")
    print(f"blk vs ref : {(out_blk - ref).abs().max().item()}")
    print(f"raw vs blk : {(out_raw - out_blk).abs().max().item()}  ← 결과는 같음")
    print("→ raw는 vectorized 1회 gather, block_ptr 우회는 BLOCK_M=32번의 직렬 1-D load")
    print("→ '직접 1:1 변환'은 불가능. 우회는 가능하지만 성능 트레이드오프 있음.")

In [ ]:
# 간단한 latency 비교 — 우회의 성능 비용 확인
if not skip_if_no_cuda():
    import time
    NUM_ROWS, D = 4096, 128
    SOURCE_ROWS = 100_000
    BLOCK_M, BLOCK_D = 64, 128
    x_src = torch.randn(SOURCE_ROWS, D, device="cuda", dtype=torch.float32)
    indices = torch.randint(0, SOURCE_ROWS, (NUM_ROWS,), device="cuda", dtype=torch.int32)
    out_raw = torch.zeros(NUM_ROWS, D, device="cuda")
    out_blk = torch.zeros(NUM_ROWS, D, device="cuda")
    grid = (triton.cdiv(NUM_ROWS, BLOCK_M),)

    def bench(fn, iters=200):
        for _ in range(10):
            fn()                                      # warmup
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / iters * 1e6  # μs

    raw_us = bench(lambda: gather_rows_raw[grid](
        x_src, indices, out_raw, NUM_ROWS, D,
        x_src.stride(0), x_src.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_D=BLOCK_D))
    blk_us = bench(lambda: gather_rows_blockptr_workaround[grid](
        x_src, indices, out_blk, NUM_ROWS, D,
        x_src.stride(0), x_src.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_D=BLOCK_D))
    print(f"raw (vectorized gather)         : {raw_us:8.2f} μs")
    print(f"block_ptr workaround (직렬 loop): {blk_us:8.2f} μs")
    print(f"우회 비용                        : {blk_us / raw_us:.2f}× slower")

## 사례 5 — 🤷 Scalar load: 변환 가능하지만 의미 없음

한 program이 lookup 테이블에서 int 1개를 읽어 곱셈 factor로 사용. raw는 한 줄. block_ptr로 표현은 가능하지만 — `shape=(1,), block_shape=(1,)`로 만들고 결과는 `[1]` 텐서. 스칼라가 필요한데 텐서가 나와서 indexing/reduction을 추가해야 함. **추상화 비용만 늘어남.**

vLLM 변환에서 `block_table` lookup, `seq_lens`, `query_start_loc`, varlen `_find_seq_idx`의 binary search 등을 의도적으로 raw로 남긴 이유.

In [ ]:
@triton.jit
def scale_by_lookup_raw(X, Lookup, Out, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    factor = tl.load(Lookup + pid)                    # 한 줄 — 스칼라
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    x = tl.load(X + offs, mask=mask, other=0.0)
    tl.store(Out + offs, x * factor, mask=mask)

@triton.jit
def scale_by_lookup_blockptr(X, Lookup, Out, N, NUM_PROGRAMS,
                             BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    # block_ptr으로 스칼라를 읽으려면 — shape=(1,) 텐서로 받아 reduce
    Lookup_ptr = tl.make_block_ptr(
        base=Lookup, shape=(NUM_PROGRAMS,), strides=(1,),
        offsets=(pid,), block_shape=(1,), order=(0,),
    )
    factor_vec = tl.load(Lookup_ptr)                  # [1] 텐서
    factor = tl.sum(factor_vec)                       # 스칼라로 만들려면 reduce 필요 (1개라 합 == 자기 자신)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    x = tl.load(X + offs, mask=mask, other=0.0)
    tl.store(Out + offs, x * factor, mask=mask)

if not skip_if_no_cuda():
    N, BLOCK = 1000, 128
    NUM_PROGRAMS = triton.cdiv(N, BLOCK)
    x = torch.randn(N, device="cuda", dtype=torch.float32)
    lookup = torch.randn(NUM_PROGRAMS, device="cuda", dtype=torch.float32)
    out_raw = torch.zeros_like(x)
    out_blk = torch.zeros_like(x)
    grid = (NUM_PROGRAMS,)
    scale_by_lookup_raw[grid](x, lookup, out_raw, N, BLOCK=BLOCK)
    scale_by_lookup_blockptr[grid](x, lookup, out_blk, N, NUM_PROGRAMS, BLOCK=BLOCK)
    ref = x.clone()
    for p in range(NUM_PROGRAMS):
        s, e = p * BLOCK, min((p + 1) * BLOCK, N)
        ref[s:e] *= lookup[p]
    print(f"raw vs ref : {(out_raw - ref).abs().max().item()}")
    print(f"blk vs ref : {(out_blk - ref).abs().max().item()}")
    print(f"raw vs blk : {(out_raw - out_blk).abs().max().item()}  ← 결과는 같음")
    print("→ block_ptr 버전은 NUM_PROGRAMS 인자 + (1,)shape + tl.sum reduce 필요")
    print("→ 표현 비용만 늘어남. scalar load는 raw가 정답.")

## 정리 — 1:1 교환 가능성 cheat sheet

| 패턴 | 1:1 교환 | 비고 |
|---|---|---|
| 단일 텐서의 contiguous tile load/store | ✅ 가능 | `boundary_check`가 `mask`의 텐서 끝 zero-pad 역할 그대로 함 |
| 2-D tile + 양축 boundary | ✅ 가능 | `boundary_check=(0, 1)` |
| K transpose | ✅ 가능 (block_ptr이 더 깨끗) | `tl.trans` 대신 `order=(0, 1)` virtual transpose |
| `tl.advance`로 다음 tile 이동 | ✅ 가능 | base 텐서가 같을 때만 |
| Mid-tensor algorithmic mask (causal, 짝수 등) | ⚠️ 분리 필요 | `boundary_check`로 대체 불가. load 후 `tl.where` 별도 단계 |
| Store에 algorithmic mask | ⚠️ 분리 필요 | block_ptr store도 `boundary_check`만 받음 |
| 한 tile 안 행마다 다른 base (gather) | ❌ **직접 1:1 불가** | `make_block_ptr`은 단일 base 필수. 우회: 행마다 fresh make_block_ptr 직렬 loop |
| paged KV의 `block_table` 간접 인덱싱 | ❌ **직접 1:1 불가** | gather와 동형. 한 iter = 한 logical block 으로 재구성하면 스칼라 base가 됨 |
| Scalar 1개 로드 (lookup, indexing 등) | 🤷 가능하지만 의미 없음 | `shape=(1,)` + `tl.sum` reduce. 추상화 비용만 추가 |
| `block_shape`에 runtime 값 사용 | ❌ 불가 | `block_shape`은 `tl.constexpr` 필수 (`shape`/`strides`/`offsets`은 runtime OK) |
| Negative/non-monotonic stride | ❌ 미지원 (Triton 제약) | block_ptr 메모리 모델은 단조 affine 가정 |

## 핵심 통찰

1. **block_ptr은 "단일 base + 정적 strides + constexpr block_shape"이 핵심 가정.** 이 셋 중 하나라도 깨지는 패턴은 직접 1:1 변환 불가.
2. **`mask`는 algorithmic + boundary 통합, `boundary_check`는 boundary만.** 책임이 분리되어 코드 단계가 늘어남.
3. **변환의 본질은 *tile* I/O 추상화.** 스칼라/gather 같이 tile 가정에 안 맞는 패턴은 raw가 정답.
4. **vLLM v1 본가는 raw pointer를 쓴다.** 사례 4에서 봤듯 paged 패턴이 block_ptr 가정에 맞지 않기 때문 — 우회는 가능하지만 vectorized gather가 더 효율적.
5. 그럼에도 학습 가치는 명확하다 — block_ptr이 **어디서 깨끗해지고, 어디서 강제로 분리되며, 어디서 막히는지**를 직접 보여주는 idiom의 카탈로그가 된다.